In [26]:
## maze_env.py

from gymnasium import spaces
import numpy as np


class MazeEnv:
    def __init__(self):
        # Grid size
        self.GRID_ROWS = 16
        self.GRID_COLS = 16

        # Walls copied from your maze layout
        self.walls = {
            (1, 1), (1, 2), (1, 3),
            (2, 3),
            (3, 1), (3, 2), (3, 3),

            (1, 6), (2, 6), (3, 6),
            (3, 5), (3, 7),
            (4, 7),

            (1, 9), (1, 10),
            (2, 9),
            (3, 9), (3, 10), (3, 11),
            (4, 11),

            (5, 1), (5, 2), (5, 3),
            (6, 3),
            (7, 1), (7, 2), (7, 3),
            (8, 1),

            (5, 5), (6, 5), (7, 5),
            (7, 6), (7, 7),
            (5, 7),

            (5, 9), (6, 9), (7, 9),
            (5, 10), (5, 11),
            (7, 11), (8, 11),

            (9, 2), (10, 2), (11, 2),
            (9, 3), (9, 4),
            (11, 4),

            (9, 6), (10, 6), (11, 6),
            (11, 7), (11, 8),
            (9, 8),

            (9, 10), (10, 10), (11, 10),
            (9, 11),
            (11, 11),

            (4, 1),
            (6, 7),
            (8, 5),
            (8, 9),
            (10, 4),
            (12, 6),
        }

        # Start and exit positions
        self.start_pos = (0, 0)
        self.exit_pos = (self.GRID_ROWS - 1, self.GRID_COLS - 1)

        # Player position
        self.player_row, self.player_col = self.start_pos

        # Episode control
        self.max_steps = 10000
        self.steps_taken = 0

        # Action space: 4 discrete moves
        # 0 = up, 1 = down, 2 = left, 3 = right
        self.action_space = spaces.Discrete(4)

        # Observation space: row and column normalized to [0, 1]
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0], dtype=np.float32),
            high=np.array([1.0, 1.0], dtype=np.float32),
            dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        self.player_row, self.player_col = self.start_pos
        self.steps_taken = 0

        obs = self._get_obs()
        info = {
            "position": (self.player_row, self.player_col),
            "distance_to_exit": self._distance_to_exit()
        }

        return obs, info

    def _get_obs(self):
        return np.array(
            [
                self.player_row / (self.GRID_ROWS - 1),
                self.player_col / (self.GRID_COLS - 1)
            ],
            dtype=np.float32
        )

    def _distance_to_exit(self):
        return abs(self.player_row - self.exit_pos[0]) + abs(self.player_col - self.exit_pos[1])

    def step(self, action):
        self.steps_taken += 1

        # Distance before movement
        old_distance = self._distance_to_exit()

        # Current position
        new_row, new_col = self.player_row, self.player_col

        # Apply action
        if action == 0:        # up
            new_row -= 1
        elif action == 1:      # down
            new_row += 1
        elif action == 2:      # left
            new_col -= 1
        elif action == 3:      # right
            new_col += 1

        moved = False

        # Move only if the new position is inside the maze and not a wall
        if (
            0 <= new_row < self.GRID_ROWS
            and 0 <= new_col < self.GRID_COLS
            and (new_row, new_col) not in self.walls
        ):
            self.player_row, self.player_col = new_row, new_col
            moved = True

        # Distance after movement
        new_distance = self._distance_to_exit()

# ---------------------------------------------------------
# Reward system
# ---------------------------------------------------------
       # Reward system
        reward = -0.1
        terminated = False
        truncated = False
        
        # Reward moving closer to the exit
        if new_distance < old_distance:
            reward += 0.3
        
        # Penalise moving farther away from the exit
        #elif new_distance > old_distance:
          #  reward -= 0.3
        
        # Strong penalty for invalid moves into walls or outside the maze
        if not moved:
            reward -= 5.0
        
        # Big reward for reaching the exit
        if (self.player_row, self.player_col) == self.exit_pos:
            reward = 100.0
            terminated = True
        
        # End episode if max step limit is reached
        if self.steps_taken >= self.max_steps:
            truncated = True

        obs = self._get_obs()

        info = {
            "old_distance": old_distance,
            "new_distance": new_distance,
            "moved": moved,
            "position": (self.player_row, self.player_col),
            "steps_taken": self.steps_taken,
            "reached_exit": (self.player_row, self.player_col) == self.exit_pos
        }

        return obs, reward, terminated, truncated, info

    def render(self):
        print(
            f"Step: {self.steps_taken} | "
            f"Position: ({self.player_row}, {self.player_col}) | "
            f"Distance to exit: {self._distance_to_exit()}"
        )


print("Done")

Done
